In [32]:
# Code to generate final Maven data element list indicating mmg and cste elements

In [33]:
import pandas as pd
import numpy as np

In [34]:
# Read all the maven elements
df_maven = pd.read_csv('input/maven_all_elements.csv',na_filter=False)
len(df_maven)

9501

In [35]:
df_maven.columns

Index(['Model', 'question_number', 'depth', 'Question', 'Question.Package.ID',
       'Question.Package.Name', 'Message'],
      dtype='object')

In [36]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If not space, digits, uppercase letters, lowercase letters, or underscore, ignore it
        if not (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=''
        else:
            new_name+=ch
    return new_name.strip()

In [37]:
# Clean up column names in both data sets
df_maven.loc[:,'Question'] =[ clean_up(q) for q in df_maven.loc[:,'Question']]

In [38]:
# Get the matches with  mmg
df_mmg_matches = pd.read_csv('output/final_mmg_matches.csv',na_filter=False)
len(df_mmg_matches)

66

In [39]:
df_mmg_matches

,final_maven_match,mmg_element,match_list,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53
0,BIRTH_DATE,Birth Date,['BIRTH_DATE','MOTHER_PREVIOUS_BIRTH_DATE','PORTAL_BIRTH_DATE','PORTAL_FLU_BIRTH_DATE','CDR_BIRTH_DATE','CONTACT_BIRTH_DATE','BIRTH_DATE','TRANSFERRED_IN_AFTER_BIRTH_DATE',...,,,,,,,,,,
1,SEX_BIRTH,Subjects Sex,,,,,,,,,...,,,,,,,,,,
2,RACE,Race Category,['RACE','RACE','RACE','RACE','RACE'],,,,...,,,,,,,,,,
3,RACE_OTHER_SPECIFY,Other Race Text,['RACE','RACE_OTHER_SPECIFY','OTHER','EAT_OTHER_MEAT_PURCHASE_DATE_TEXT','EAT_OTHER_POULTRY_PURCHASE_DATE_TEXT','DRINK_OTHER_MILK_PURCHASE_DATE_TEXT','EAT_OTHER_DAIRY_PURCHASE_DATE_TEXT','OTHER_RISK_EXPOSURE_SHELLFISH_EXPOSURE_DATE_...,...,,,,,,,,,,
4,CLIENT_ETHNICITY,Ethnic Group,,,,,,,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,REPORTING_STATE,Reporting State,['REPORTING_SOURCE_FACILITY_NAME_STATE','REPORTING_SOURCE_FACILITY_NAME_OTHER_STATE','REPORTING_SOURCE_AGENCY_STATE','REPORTING_NOTES','REPORTING_STATE','STATE','STATE','STATE',...,,,,,,,,,,
62,REPORT_FACILITY_COUNTY,Reporting County,['COUNTY','COUNTY','COUNTY','COUNTY','FR_COUNTY','IX_COUNTY','COUNTY','COUNTY',...,,,,,,,,,,
63,JURISDICTION,National Reporting Jurisdiction,['JURISDICTION','JURISDICTION'],,,,,,,...,,,,,,,,,,
64,COMMENTS,Comment,['VACCINE_HISTORY_COMMENT','EDRS_COMMENT','COMMENTS','COMMENTS'],,,,,...,,,,,,,,,,


In [40]:
# Column 1 contains mmg element name
df_mmg_matches.iloc[0,1]

'Birth Date'

In [41]:
# Create a column in df_maven for mmg_element
num_matches=0
df_maven['mmg_element']=''
# Match the maven elements in df_maven and df_mmg_matches and place the corresponding mmg_element
match_list = df_mmg_matches.loc[:,'final_maven_match']
# For each match in column final_maven_match 
for i in range(len(match_list)):
    mmg_element = df_mmg_matches.iloc[i,1]
    tokens = match_list[i].split(',')
    flag=0
    for token in tokens:
        token = token.strip()
        # Filter df_maven to get match with token
        df_sub = df_maven[df_maven['Question']==token]
        if len(df_sub)>0:
            flag=1
        # Place mmg element in that row
        for idx in df_sub.index:
            df_maven.loc[idx,'mmg_element'] = mmg_element
    if flag==1:
        num_matches+=1
    else:
        print('No matches for',mmg_element)
            
print('Num of matches',num_matches)

        

No matches for Country of Exposure
No matches for Binational Reporting Criteria
No matches for Local Subject ID
Num of matches 63


In [42]:
# Get the matches with cste
df_cste_matches = pd.read_csv('output/final_cste_matches.csv',na_filter=False)
len(df_cste_matches)

62

In [43]:
# column 1 contains cste element name
df_cste_matches.iloc[0,1]

'Discharge Date'

In [44]:
# Create a column in df_maven for mmg_element
num_matches=0
df_maven['cste_element']=''
# Match the maven elements in df_maven and df_cste_matches and place the corresponding cste_element
match_list = df_cste_matches.loc[:,'final_maven_match']
# For each match in column final_maven_match 
for i in range(len(match_list)):
    cste_element = df_cste_matches.iloc[i,1]
    tokens = match_list[i].split(',')
    flag=0
    for token in tokens:
        token = token.strip()
        # For tokens that can have multiple matches
        if '*' in token:
            token= token.strip('*')
            df_sub = df_maven[df_maven.Question.str.contains(token)]
        else:
            # Filter df_maven to get match with token
            df_sub = df_maven[df_maven['Question']==token]
        if len(df_sub)>0:
            flag=1
        # Place cste element in that row
        for idx in df_sub.index:
            df_maven.loc[idx,'cste_element'] = cste_element
    if flag==1:
        num_matches+=1
    else:
        print('No matches for',cste_element)

print('Num of matches',num_matches)


No matches for Death Associated with Condition
No matches for Housing Status
No matches for Primary Living Situation
No matches for Disability Type
No matches for Disability Type Indicator
No matches for Summative Exposure Indicator
No matches for Summative Underlying Health Condition Indicator
No matches for Admitted to Intensive Care Unit ICU
No matches for Identifier Type
No matches for Identifier Value
No matches for Specimen Type
No matches for Medication Status
No matches for Date Medication Prescribed
No matches for Medication Prescribed Duration
No matches for Date Medication Started
No matches for Date Medication Ended
No matches for Specify Different Travel Exposure Window
Num of matches 45


In [45]:
df_maven.to_csv('output/maven_all_with_mmg_and_cste.csv',index=False)